# Regresion Lineal V. `Scikit-Learn`. 

Hasta ahora hemos implementado nuestras propias funciones para aplicar *Regresión Lineal* o *Regresión Polinómica*.
Vamos a ver en este Notebook cómo usar `scikit-learn` para realizar regresiones lineales y polinómicas, dividir en *Test* y *Train*, escalar datos y aplicar regularización, todo sin la necesidad de implementar manualmente cada método.

In [ ]:
# Importar las bibliotecas necesarias
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score


import seaborn as sns
import matplotlib.pyplot as plt
import random

Vamos a usar `sklearn` para generar de forma sintética los datos de entrada al modelo.
De momento empleamos el método [`datasets.make_regression`](/https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_regression.html) para generar de forma aleatoria una matriz de entrada con 100 registros con 6 características de las cuales solo 4 aportan información.

Para dividir los datos de entrada entre *Train* y *Test* con la estrategia de *Hold-out 80-20%* empleamos [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html). 

In [ ]:
X, y = datasets.make_regression(n_samples=100, n_features=6, n_informative=4, noise=20, random_state=4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

## Regresión Lineal.

[`LinearRegression()`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) implementa la **Ecuación Normal**
* `lin_reg = LinearRegression()`: crea un objeto de la clase `LinearRegression()` que se utiliza para generar la aproximación mediante regresión lineal. Al llamarlo, se crea una estructura que puede ser utilizada más adelante para ajustar un modelo a los datos.

* `lin_reg.fit(X_train, y_train)`: -**Entrenamiento**- Esta línea ajusta el modelo de regresión lineal a los datos de entrenamiento. Utiliza el método *fit* de la clase `LinearRegression()` para calcular los coeficientes del modelo basándose en los datos proporcionados.

* `y_pred_lin = lin_reg.predict(X_test)`: -**Predicción**- A partir del modelo ajustado calcula predicciones sobre un nuevo conjunto de datos (en este caso, el conjunto de prueba). Emplea el método *predict* de la clase `LinearRegression()`


`y_pred_lin` contiene ahora la expresión conocida:
$$ 
\hat{y}^j = h_\Theta(\boldsymbol{x^j})= \theta_0 + \theta_1 x^j_1 + \theta_2 x^j_2 + \dots +\theta_n x^j_n 
$$


Para calcular los coeficientes, usa internamente:  

$$
\Theta = (X^TX)^{-1}X^TY
$$

pero no lo hace literalmente porque esa forma es numéricamente inestable si $X^TX$ está mal condicionada.
En su lugar, aplica SVD de la matriz de diseño $X$, lo que es más robusto y evita problemas de multicolinealidad. [Documentación de Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)



En los modelos de regresión en `sklearn` los coeficientes $\Theta = \left( \theta_0, \theta_1, \dots, \theta_n \right)$ se devuelven en dos estructuras:
* `lin_reg.intercept_`: devuelve $\theta_0$, también llamado intercepto o sesgo (*Bias*).
* `lin_reg.coef_`: devuelve el resto de pesos del vector de coeficientes $\Theta$, desde $\left( \theta_1, \theta_2, \dots, \theta_n \right)$





In [ ]:
# --- Regresión Lineal ---
# Ajuste del modelo de regresión lineal
lin_reg = LinearRegression()
# Entrenamiento del modelo
lin_reg.fit(X_train, y_train)

# Resultados de la regresión lineal
print("Regresión Lineal:")
print("Coeficientes:", lin_reg.coef_)
print("Intercepto:", lin_reg.intercept_)

# Predicción
y_pred_train = lin_reg.predict(X_train)
y_pred_test = lin_reg.predict(X_test)


#### Métricas para el error.

Como métricas para evaluar el rendimiento de nuestro modelo empleamos:

*  El Error Cuadrático Medio (MSE): Como ya sabemos el MSE es una medida que calcula la media de los cuadrados de los errores, donde el error es la diferencia entre los valores predichos por el modelo y los valores reales.

* Coeficiente de Determinación ($R^2$): Es una medida que indica la proporción de la variabilidad total de la variable dependiente que es explicada por el modelo de regresión. Es un valor entre 0 y 1.
     *  Un $R^2=0$ indica que el modelo no explica nada de la variabilidad en los datos.
     *  Un $R^2=1$ indica que explica toda la variabilidad.

Para evaluar ambas métricas contamos con las funciones `mean_squared_error` y `r2_score` disponibles en `sklearn.metrics`.


In [ ]:
# Cálculo de métricas para la regresión lineal
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)


print("Métricas de rendimiento en el conjunto de entrenamiento:")
print(f"  MSE (Train): {mse_train:.4f}")
print(f"  R² (Train): {r2_train:.4f}")

print("\nMétricas de rendimiento en el conjunto de test:")
print(f"  MSE (Test): {mse_test:.4f}")
print(f"  R² (Test): {r2_test:.4f}")

## Regresión Lineal con Descenso de Gradiente Estocástico.

Usamos [`SGDRegressor()`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDRegressor.html) 
La llamada `sgd_reg = SGDRegressor(max_iter=1000, tol=1e-3, eta0=0.01, random_state=1234)` crea un objeto de la clase `SGDRegressor()` que se utiliza para implementar la regresión lineal usando descenso de gradiente estocástico para calcular los pesos. 

* `max_iter=1000`:  El hiperparámetro max_iter=1000 controla el número máximo de iteraciones/epochs.

* `tol=1e-3`: establece el criterio de tolerancia para la convergencia.

* `eta0=0.01`: Tasa de aprendizaje inicial. Por defecto irá descendiendo para ajustarse mejor al óptimo.


In [ ]:
# Iniciar el modelo SGDRegressor
sgd_reg = SGDRegressor(max_iter=1000, tol=1e-3, eta0=0.01, random_state=1234)

# Entrenar el modelo
sgd_reg.fit(X_train, y_train)

# Resultados de la regresión lineal
print("Regresión Lineal:")
print("Coeficientes:", sgd_reg.coef_)
print("Intercepto:", sgd_reg.intercept_)

# Predicciones
y_pred_train = sgd_reg.predict(X_train)
y_pred_test = sgd_reg.predict(X_test)


#### Métricas para el error.

In [ ]:
# Calcular métricas de rendimiento (MSE y R²) para el SGD
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

# Mostrar resultados
print("Métricas de rendimiento en el conjunto de entrenamiento:")
print(f"  MSE (Train): {mse_train:.4f}")
print(f"  R² (Train): {r2_train:.4f}")

print("\nMétricas de rendimiento en el conjunto de test:")
print(f"  MSE (Test): {mse_test:.4f}")
print(f"  R² (Test): {r2_test:.4f}")

## Regresión Polinómica.



#### Generación de datos a partir de función polinómica.

Vamos a generar un conjunto de datos en el intervalo $[0, 10]$ a partir del polinomio

$ f(x) = 0.5x^5 - 2x^4 + x^3 +x^2-x+10 $

El *dataset* estará formado por 30 puntos de la forma $X = \{(x_i, f(x_i) + \epsilon) \}$ donde $\epsilon$ será una variable aleatoria que seguirá una distribución normal de media 0 y desviación típica 200.

In [ ]:
# 1. Data Generation: Creating a polynomial of degree 5 with noise
np.random.seed(42)
n_samples = 30
X = np.random.rand(n_samples, 1) * 10  # Random values between 0 and 10
y_true = 0.5 * X**5 - 2*X**4 + X**3 + X**2 - X + 10  # True polynomial of degree 5
noise = np.random.normal(0, 200, size=y_true.shape)  # Add Gaussian noise
y = y_true + noise

#### División en *Train-Test*

In [ ]:
# 2. Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Generación de la matriz $\boldsymbol{X}$ de potencias sucesivas.

Empleamos `PolynomialFeatures(degree=degree)` para transformar los datos de *Train* y *Test* para adaptarlos al orden polinómico deseado antes de generar el Regresor Lineal `lin_reg = LinearRegression()` sobre los ordenes polinómicos creados.

[`PolynomialFeatures(degree)`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) crea las sucesivas potencias de la variable original hasta el grado deseado. 

Se puede emplear si el conjunto de datos de entrada tiene más de una característica. Por ejemplo, si la entrada es un conjunto de datos con dos variables $\left\{ x_1, x_2 \right\}$, para `degree=2`, devolverá como características del conjunto de datos de entrada $\left\{1, x_1, x_2, x_1^2, x_2x_1, x_2^2\right\}$

* La función `.fit_transform(X_train)` se aplica sobre el conjunto de puntos de entrenamiento. Sirve para crear una estructura basada en el número de características de los datos y generar las combinaciones polinómicas según el grado.
* La función `.transform(X_test)` aplica la transformación previamente aprendida al conjunto de test.

In [ ]:
degree = 24
poly_features = PolynomialFeatures(degree)
X_train_poly = poly_features.fit_transform(X_train)
X_test_poly = poly_features.transform(X_test)

# Ajuste del modelo de regresión lineal
lin_reg = LinearRegression()
lin_reg.fit(X_train_poly, y_train)
    
# Predictions and evaluation
y_train_pred = lin_reg.predict(X_train_poly)
y_test_pred = lin_reg.predict(X_test_poly)

# Resultados de la regresión polinómica
print("Regresión Polinómica:")
print("Coeficientes:", lin_reg.coef_)
print("Intercepto:", lin_reg.intercept_)

#### Métricas del error.

In [ ]:
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Degree: {degree}")
print(f"Train MSE: {train_mse:.3f}, Test MSE: {test_mse:.3f}")
print(f"Train R^2: {train_r2:.3f}, Test R^2: {test_r2:.3f}")
print("-"*40)

## Regularización.

Para implementar regularización en modelos de regresión lineal utilizando `sklearn`, se pueden usar las clases correspondientes a Ridge (para la regularización L2) y Lasso (para la regularización L1).
Volvemos al ejemplo sintético ya usado.

In [ ]:
X, y = datasets.make_regression(n_samples=100, n_features=6, n_informative=4, noise=20, random_state=4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

#### Escalado previo de los datos.

Antes de aplicar regularización es fundamental escalar los datos. Usamos [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#### Regularización de Ridge.

También conocido como Regresión Ridge o regularización de Tikhonov.
La función [`Ridge()`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html#sklearn.linear_model.Ridge) de la clase `sklearn.linear_model.Ridge` implementa la regresión lineal con regularización L2, penalizando el tamaño de los coeficientes de las características para reducir el riesgo de overfitting. 

Dependiendo del tamaño del conjunto de datos y la configuración del parámetro solver puede elegir el método de optimización que utilizará. Entre los métodos están:
* Ecuación normal (`solver='svd'` o `solver='cholesky'`)
* Descenso de gradiente (Stochastic Gradient Descent, SGD) (`solver='sag'` o `solver='saga'`) 
* `auto` (valor por defecto): El algoritmo selecciona automáticamente el mejor método en función del tamaño del dataset y otras características.

Ejemplo:
``` python
ridge_model = Ridge(alpha=1.0, solver='sag')
```

El parametro `alpha` es $\lambda$ en lo que hemos visto en teoría. Controla el grado de regularización. $\lambda = 0$ implica sin regularización.

In [ ]:
ridge_reg = Ridge(alpha=1.0)  # Alpha controla la intensidad de la regularización
ridge_reg.fit(X_train_scaled, y_train)
y_pred_ridge = ridge_reg.predict(X_test_scaled)

####  Métricas de rendimiento para Ridge

In [ ]:
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)
print(f"Ridge Regression - MSE: {mse_ridge:.4f}, R²: {r2_ridge:.4f}")

#### Regularización de Lasso.

También llamada Lasso Regression, donde Lasso significa *Least Absolute Shrinkage and Selection Operator*
La función [`Lasso()`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html#sklearn.linear_model.Lasso) de la clase `sklearn.linear_model.Lasso` implementa la regresión lineal con regularización L1, penalizando el tamaño de los coeficientes de las características para reducir el riesgo de overfitting. 

Al igual que con Ridge, el algoritmo que usa `Lasso()` depende del tamaño del conjunto de datos y el solver que se elija.
* `auto`: Selecciona automáticamente el mejor solver basado en las características de los datos.
* `cd`: El descenso en coordenadas (Coordinate Descent) es el método predeterminado. Es adecuado para conjuntos de datos pequeños a moderados.
* `saga`: Utiliza descenso de gradiente estocástico mejorado para la regularización L1, más eficiente en grandes conjuntos de datos.

Ejemplo:
``` python
Lasso_model = Lasso(alpha=1.0, solver='saga')
```

El parametro `alpha` es $\lambda$ en lo que hemos visto.

Vamos a partir del conjunto de datos anterior ya escalado.

In [ ]:
lasso_reg = Lasso(alpha=0.1)  # Alpha controla la intensidad de la regularización
lasso_reg.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_reg.predict(X_test_scaled)

#### Métricas de rendimiento para Lasso.

In [ ]:
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

print(f"Lasso Regression - MSE: {mse_lasso:.4f}, R²: {r2_lasso:.4f}")

### Efecto de la normalización en la Regularización

Para ver el efecto de la normalización/escalado de las variables de entrada antes de aplicar regularización vamos a trabajar con un caso de Regresión Polinómica. Al ser las columnas de la matriz de datos potencias sucesivas de la variable de entrada, en cuanto el orden polinómico alcanzado sea alto, nuestra matriz tendrá valores con ordenes de magnitud muy diferente.

#### Generación de datos a partir de función polinómica.

Vamos a generar un conjunto de datos en el intervalo $[-3, 3]$ a partir del polinomio

$ f(x) = 0.5x^5 - x^4 + x^3 - x^2+x $

El *dataset* estará formado por 150 puntos de la forma $X = \{(x_i, f(x_i) + \epsilon) \}$ donde $\epsilon$ será una variable aleatoria que seguirá una distribución normal de media 0 y desviación típica 40.

In [ ]:
np.random.seed(42)
X = 6 * np.random.rand(150, 1) - 3  # Genero datos entre [-2, 4]
y = 0.5 * X**5 - X**4 + X**3 - X**2 + X + np.random.randn(150, 1) * 40  # Polinomio de grado 5 con ruido

#### Dividir en conjunto de entrenamiento y prueba

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Creación de la matriz $\boldsymbol{X}$ de potencias sucesivas.

Vamos a llegar a orden polinómico 12 para ver mejor el efecto

In [ ]:
# Generar características polinómicas de grado 12
poly = PolynomialFeatures(degree=12, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

#### Comparación resultados con y sin escalado.

Usamos la regularización de **Ridge** para ver el efecto de escalar previamente los datos de entrada. 

In [ ]:
# Ajuste con Ridge sin escalado
ridge_no_scaling = Ridge(alpha=1.0)
ridge_no_scaling.fit(X_train_poly, y_train)
y_pred_no_scaling = ridge_no_scaling.predict(X_test_poly)

# Escalado de las características polinómicas
scaler = StandardScaler()
X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

# Ajuste con Ridge con escalado
ridge_scaling = Ridge(alpha=1.0)
ridge_scaling.fit(X_train_poly_scaled, y_train)
y_pred_scaling = ridge_scaling.predict(X_test_poly_scaled)

# Gráfica de predicciones vs. valores reales
plt.figure(figsize=(12, 6))

# Sin escalado
plt.subplot(1, 2, 1)
plt.scatter(X_test, y_test, label='Valores Reales', color='b')
plt.plot(np.sort(X_test, axis=0).ravel(), y_pred_no_scaling[np.argsort(X_test, axis=0).ravel()], color='r', label='Predicción Sin Escalar')
plt.title('Ridge sin Escalar (Polinomio Grado 12)')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()

# Con escalado
plt.subplot(1, 2, 2)
plt.scatter(X_test, y_test, label='Valores Reales', color='b')
plt.plot(np.sort(X_test, axis=0).ravel(), y_pred_scaling[np.argsort(X_test, axis=0).ravel()], color='r', label='Predicción Escalada')
plt.title('Ridge con Escalado (Polinomio Grado 12)')
plt.xlabel('X Escalado')
plt.ylabel('y')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Obtener los coeficientes de la regresión
coeficientes_Ridge = ridge_scaling.coef_
intercepto_Ridge = ridge_scaling.intercept_

coeficientes_Ridge_no_esc = ridge_no_scaling.coef_
intercepto_Ridge_no_esc = ridge_no_scaling.intercept_


# Mostrar los coeficientes
print("Coeficientes de la regresión por Ridge con escalado:",intercepto_Ridge, coeficientes_Ridge)
print("Coeficientes de la regresión por Ridge sin escalado:",intercepto_Ridge_no_esc, coeficientes_Ridge_no_esc)

Se puede apreciar la necesidad de escalar en la gráfica donde el modelo sin escalado pierde capacidad de predicción y además tiende a sobreajustar a los datos de entrenamiento *overfitting*.
Además si aumentamos el orden polinómico podemos tener problemas de estabilidad numérica.

Para grado 25 y al reducir el ruido a $\sigma = 5$ los datos están próximos entre ellos y la ejecución muestra la siguiente advertencia por matriz mal condicionada:
```python 
LinAlgWarning: Ill-conditioned matrix (rcond=6.2944e-25): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
```

#### Comparación de los coeficientes de Lasso y Ridge.

Como hemos dicho **Ridge** (L2) penaliza el tamaño de los coeficientes y tiende a reducirlos sin llevarlos a cero mientras que **Lasso** (L1) puede reducir los coeficientes a exactamente cero, realizando una forma de selección de características.
Veamos como son los coeficientes en este último ejemplo donde estamos usando un polinomio de grado 25 para aproximar una función de grado 5 con ruido.

In [ ]:
# Obtener los coeficientes de la regresión
coeficientes_Ridge = ridge_scaling.coef_
intercepto_Ridge = ridge_scaling.intercept_

# Mostrar los coeficientes
print("Coeficientes de la regresión por Ridge:",intercepto_Ridge, coeficientes_Ridge)

# Ajuste con Lasso con escalado
lasso_scaling = Lasso(alpha=1.0)
lasso_scaling.fit(X_train_poly_scaled, y_train)
y_pred_scaling_Lasso = lasso_scaling.predict(X_test_poly_scaled)
# Obtener los coeficientes de la regresión
coeficientes_Lasso = lasso_scaling.coef_
intercepto_Lasso = lasso_scaling.intercept_

# Mostrar los coeficientes
print("Coeficientes de la regresión por Lasso:",intercepto_Lasso, coeficientes_Lasso)



In [ ]:
# Gráfica de predicciones vs. valores reales
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(X_test, y_test, label='Valores Reales', color='b')
plt.plot(np.sort(X_test, axis=0).ravel(), y_pred_scaling[np.argsort(X_test, axis=0).ravel()], color='r', label='Predicción Escalada')
plt.title('Ridge con Escalado (Polinomio Grado 12)')
plt.xlabel('X Escalado')
plt.ylabel('y')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(X_test, y_test, label='Valores Reales', color='b')
plt.plot(np.sort(X_test, axis=0).ravel(), y_pred_scaling[np.argsort(X_test, axis=0).ravel()], color='r', label='Predicción Escalada')
plt.title('Lasso con Escalado (Polinomio Grado 12)')
plt.xlabel('X Escalado')
plt.ylabel('y')
plt.legend()

plt.tight_layout()
plt.show()


El resultado es muy similar pero como ya sabíamos **Lasso** ha forzado que los coeficientes $>5$ sean exactamente cero, realizando una forma de selección de características.